## 📦 1. IMPORTS + SETUP

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/evgendvorkin/dvorkin-visual-v1/')
from Dvorkin_style_engine import *
set_theme("cosmic")  # или "gold", "ocean", "forest" - на твой вкус

import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

T0 = time.perf_counter()
def log(msg):
    print(f'[{time.perf_counter() - T0:7.1f}s] {msg}', flush=True)

h1("🚗 EV Adoption Prediction — S6E9")
info_card("✅ Setup Complete", "Импорты выполнены, стиль загружен.", style="vi")

## 📥 2. LOAD DATA

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv')
orig = pd.read_csv('/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/EV_Adoption_and_Range_Anxiety_Dataset.csv')

h2("📥 Data Loading")
info_card("Dataset Shapes", 
          f"Train: {train.shape} | Test: {test.shape} | Orig: {orig.shape}", 
          style="vi")
pretty_df(train.head(3))

## 🔍 3. EDA — Exploratory Data Analysis


In [ ]:
def analyze_datasets(train_df, test_df, orig_df):
    """
    Комплексный анализ трёх датасетов с красивым выводом.
    """
    datasets = {
        'train': train_df,
        'test': test_df,
        'original': orig_df
    }
    
    MAX_CAT_VALUES = 10
    
    for name, df in datasets.items():
        h2(f"{name.upper()} Dataset")
        info_card("Shape", f"Rows: {len(df)} | Columns: {len(df.columns)}", style="vi")
        
        # Missing values
        missing = df.isnull().sum()
        total_missing = missing.sum()
        
        if total_missing == 0:
            info_card("✅ Missing Values", "Нет пропусков", style="vi")
        else:
            missing_df = missing[missing > 0].reset_index()
            missing_df.columns = ['Column', 'Missing Count']
            missing_df['% Missing'] = (missing_df['Missing Count'] / len(df) * 100).round(2)
            info_card("⚠️ Missing Values", f"Всего пропусков: {total_missing} ({total_missing/df.size*100:.2f}% всех ячеек)", style="co")
            pretty_df(missing_df)
        
        # Numerical Features
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        
        if num_cols:
            h2(f"📈 Numerical Features ({len(num_cols)})")
            desc = pd.DataFrame({
                'Column': num_cols,
                'Mean': [df[col].mean() for col in num_cols],
                'Std': [df[col].std() for col in num_cols],
                'Min': [df[col].min() for col in num_cols],
                '25%': [df[col].quantile(0.25) for col in num_cols],
                '50%': [df[col].quantile(0.5) for col in num_cols],
                '75%': [df[col].quantile(0.75) for col in num_cols],
                'Max': [df[col].max() for col in num_cols],
                'Skew': [stats.skew(df[col].dropna()) for col in num_cols]
            })
            pretty_df(desc)
            
            # Выбросы (IQR)
            outlier_rows = []
            for col in num_cols:
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
                outlier_rows.append({
                    'Column': col,
                    'Outliers': len(outliers),
                    'Lower Bound': round(lower_bound, 2),
                    'Upper Bound': round(upper_bound, 2)
                })
            info_card("🔍 Outliers (IQR)", "Ниже — таблица выбросов для числовых признаков", style="cy")
            pretty_df(pd.DataFrame(outlier_rows))
        
        # Categorical Features
        if cat_cols:
            h2(f"📋 Categorical Features ({len(cat_cols)})")
            cat_info = pd.DataFrame({
                'Column': cat_cols,
                'Unique Values': [df[col].nunique() for col in cat_cols]
            })
            pretty_df(cat_info)
            
            # Для каждой категории выводим топ
            for col in cat_cols:
                vc = df[col].value_counts().head(MAX_CAT_VALUES)
                if len(vc) > MAX_CAT_VALUES:
                    info_card(f"📊 {col}", f"Показаны топ {MAX_CAT_VALUES} из {df[col].nunique()} значений", style="cy")
                else:
                    info_card(f"📊 {col}", f"Всего {df[col].nunique()} уникальных значений", style="cy")
                pretty_df(pd.DataFrame({
                    'Value': vc.index,
                    'Count': vc.values,
                    '%': (vc.values / len(df) * 100).round(2)
                }))

# Запуск анализа
analyze_datasets(train, test, orig)

## 🔧 4. PREPROCESSING (BASE)


In [ ]:
cat_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
            'Subsidy_Available', 'Range_Anxiety_Level']

for col in cat_cols:
    mapping = {val: i for i, val in enumerate(train[col].unique())}
    train[f'LE_{col}'] = train[col].map(mapping)
    test[f'LE_{col}'] = test[col].map(mapping)

# Удаляем исходные категориальные колонки
train = train.drop(columns=cat_cols)
test = test.drop(columns=cat_cols)

# Убираем id
train = train.drop(columns=['id'])
test = test.drop(columns=['id'])

# Разделяем X и y
X = train.drop(columns=['Will_Buy_EV'])
y = train['Will_Buy_EV']

h2("🔧 Preprocessing")
info_card("✅ Base Preprocessing Done", f"X shape: {X.shape} | Test shape: {test.shape}", style="vi")

## 🧪 5. FEATURE ENGINEERING — Powerful Interactions


In [ ]:
# Берём исходные данные (train_cat уже есть из ячейки 4)
X['Env_Concern_x_Subsidy'] = X['Environmental_Concern_Level'] * X['LE_Subsidy_Available']
test['Env_Concern_x_Subsidy'] = test['Environmental_Concern_Level'] * test['LE_Subsidy_Available']

X['Env_Concern_x_Income'] = X['Environmental_Concern_Level'] * X['Annual_Income_USD']
test['Env_Concern_x_Income'] = test['Environmental_Concern_Level'] * test['Annual_Income_USD']

X['Subsidy_x_Income'] = X['LE_Subsidy_Available'] * X['Annual_Income_USD']
test['Subsidy_x_Income'] = test['LE_Subsidy_Available'] * test['Annual_Income_USD']

X['Env_Concern_x_Commute'] = X['Environmental_Concern_Level'] * X['Daily_Commute_km']
test['Env_Concern_x_Commute'] = test['Environmental_Concern_Level'] * test['Daily_Commute_km']

X['Home_Charging_x_Subsidy'] = X['LE_Home_Charging_Possible'] * X['LE_Subsidy_Available']
test['Home_Charging_x_Subsidy'] = test['LE_Home_Charging_Possible'] * test['LE_Subsidy_Available']

X['Range_Anxiety_x_Subsidy'] = X['LE_Range_Anxiety_Level'] * X['LE_Subsidy_Available']
test['Range_Anxiety_x_Subsidy'] = test['LE_Range_Anxiety_Level'] * test['LE_Subsidy_Available']

# Ratio с главными признаками
X['Income_per_Concern'] = X['Annual_Income_USD'] / (X['Environmental_Concern_Level'] + 1)
test['Income_per_Concern'] = test['Annual_Income_USD'] / (test['Environmental_Concern_Level'] + 1)

X['Commute_per_Concern'] = X['Daily_Commute_km'] / (X['Environmental_Concern_Level'] + 1)
test['Commute_per_Concern'] = test['Daily_Commute_km'] / (test['Environmental_Concern_Level'] + 1)

print(f"✅ Добавлено interaction признаков")
print(f"X shape: {X.shape}")

## 6. 🔢 DIGIT FEATURES — ловим артефакты синтетики

**What are digit features?**  
We extract each digit of numerical features at positions -4 to 3: `digit_k(x) = (x // 10^k) % 10`.  
For 13 numeric columns this gives 104 lightweight int8 features.

**Why they work:**  
Synthetic data often contains artifacts in digits (repeated decimals, rounding patterns). Trees struggle to capture these micro-patterns from raw values; explicit digits make them usable.

In [ ]:
# Числовые признаки для извлечения цифр
digit_num_cols = [
    'Age', 'Annual_Income_USD', 'Daily_Commute_km',
    'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work', 'Environmental_Concern_Level'
]

# Извлекаем цифры в позициях от -4 до 3
for col in digit_num_cols:
    for k in range(-4, 4):
        new_col = f'{col}_digit{k}'
        X[new_col] = (X[col].fillna(0) // (10**k) % 10).astype('int8')
        test[new_col] = (test[col].fillna(0) // (10**k) % 10).astype('int8')

print(f"✅ Добавлено digit features: {len(digit_num_cols) * 8}")
print(f"X shape: {X.shape} | Test shape: {test.shape}")

## 7. 📊 FREQUENCY ENCODING — частоты для ВСЕХ признаков

**What:**  
Each value in every column is replaced by its normalized frequency across the combined train and test sets. Rare values get small frequencies, common values get large frequencies.

**Why it works:**  
It gives XGBoost a different view of the data: how unusual or typical a value is. In synthetic datasets, generators often leave unusual distributions or rare combinations. Frequency features expose these patterns directly, without many sequential tree splits.

**Effect:**  
OOF AUC improved from 0.94347 to 0.94466.

In [ ]:
# Считаем частоты по train+test для каждой колонки (включая digit и LE)
for column in X.columns:
    freq_map = pd.concat([X[column], test[column]], axis=0).value_counts(normalize=True).to_dict()
    X[f'{column}_freq'] = X[column].map(freq_map).astype('float32').values
    test[f'{column}_freq'] = test[column].map(freq_map).astype('float32').values

print(f"✅ Добавлено frequency features: {len([c for c in X.columns if c.endswith('_freq')])}")
print(f"X shape: {X.shape} | Test shape: {test.shape}")

## 🚀 8. XGBOOST — BASELINE MODEL (10-fold, dynamic seeds)

🎯 Validation Strategy — 10-Fold CV with Dynamic Seeds

We use **10-fold stratified cross-validation** to get a more stable and honest estimate of model performance.  
On each fold, the random seed is **different** (`42 + fold`), which creates additional diversity across models and reduces the CV–LB gap.

| Setting | Value |
|---------|-------|
| Folds | 10 |
| Split type | StratifiedKFold |
| Shuffle | True |
| Base seed | 42 |
| Per-fold seed | 42 + fold (43, 44, ..., 52) |

This strategy is deliberately stronger than a single-seed 5-fold setup:  
- more training data per fold (90% train / 10% valid),  
- more robust OOF estimate,  
- lower variance across runs,  
- better generalization on the private leaderboard.

We trust **OOF AUC** as the main optimization target — public LB is treated as a noisy sample.

In [ ]:
import xgboost as xgb

# Конвертируем таргет в 0/1
y = y.map({'No': 0, 'Yes': 1})

# Параметры: старые + 10 фолдов + динамические сиды
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'learning_rate': 0.03,
    'max_depth': 7,
    'min_child_weight': 5,
    'subsample': 0.7,
    'colsample_bytree': 0.5,
    'n_estimators': 2000,
    'early_stopping_rounds': 100,
    'nthread': -1,
    'deterministic_histogram': True,
}

# StratifiedKFold — 10 фолдов
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Массивы для сохранения
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))
feature_importance = np.zeros(X.shape[1])

log("Starting 10-fold CV...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Динамический seed: 42 + fold (43, 44, 45...)
    current_seed = 42 + fold
    params['random_state'] = current_seed
    params['seed'] = current_seed
    
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(test)[:, 1] / skf.n_splits
    
    feature_importance += model.feature_importances_ / skf.n_splits
    
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    h2(f"Fold {fold}/10")
    md_metric("Fold AUC", f"{fold_auc:.5f}")
    md_metric("Seed", str(current_seed))
    md_metric("Elapsed", f"{time.perf_counter() - T0:.1f}s")

# Overall OOF score
oof_auc = roc_auc_score(y, oof_preds)
h2("🏆 Final OOF Performance")
md_metric("OOF AUC", f"{oof_auc:.5f}")

# Feature importance
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

h2("📊 Top 15 Features")
pretty_df(importance_df.head(15))

| Metric | Before digit | After digit | Gain |
|--------|--------------|-------------|------|
| OOF AUC | 0.94204 | **0.94347** | **+0.00143** |

Extracting 104 digit features from 13 numerical columns helped the model capture synthetic generation artifacts. XGBoost now uses many new uncorrelated signals, improving overall ranking.

📊 Impact of Frequency Encoding

OOF AUC improved from **0.94347** to **0.94466** (+0.00119).

| Metric | Before Frequency | After Frequency | Gain |
|--------|------------------|-----------------|------|
| OOF AUC | 0.94347 | **0.94466** | **+0.00119** |

Frequency encoding adds the normalized occurrence rate of each value across train+test. This helps the model identify rare and common patterns, and exposes subtle distributional signals in numerical and digit features that raw values do not reveal. XGBoost now relies on a richer, more diverse representation.

## 💾 9. SAVE PREDICTIONS

In [ ]:
np.save('oof_preds_base.npy', oof_preds)
np.save('test_preds_base.npy', test_preds)

h2("💾 Save Predictions")
info_card("✅ Predictions Saved", 
          f"OOF: {oof_preds.shape} | Test: {test_preds.shape} | OOF AUC: {oof_auc:.5f}", 
          style="vi")

## 📤 10. CREATE SUBMISSION


In [ ]:
submission = pd.DataFrame({
    'id': pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv')['id'],
    'Will_Buy_EV': test_preds
})

submission.to_csv('submission.csv', index=False)

h2("📤 Submission Created")
info_card("✅ Submission Saved", f"Shape: {submission.shape}", style="vi")
pretty_df(submission.head(10))

### 📋 8. FUTURE PLAN

# 🎯 Roadmap for Model Improvement

### ✅ Completed
- Baseline XGBoost model (5-fold CV)
- AUC: 0.94184 (OOF) / 0.94169 (LB)
- Label Encoding for categorical features
- Saved OOF and test predictions

### 🔥 Next Steps

### Level 2 — Stacking
- Train XGBoost Level 2 on [base features + OOF from Level 1]
- Blend: Level 1 (50%) + Level 2 (50%)

### Feature Engineering
- **Combination Features**: pairs, triples, quadruples of categorical features
- **Target Encoding**: for categories and combinations (KFold anti-overfitting)
- **Ratio Features**: Income/Age, Stations/Commute, Income/Car
- **Aggregation Features**: mean/std by groups (City, Car Type, Gender)
- **Binned Features**: qcut for numerical features
- **Digit Features**: extract digits from numbers (generator artifacts)
- **Benford's Law**: deviation of first digit distribution

### Original Dataset
- **Original Data as Features**: Target Encoding from original dataset
- **Snap Features**: nearest neighbors to original (cKDTree)
- **Original Statistics**: group means from original dataset

### Models
- LightGBM (with same features)
- CatBoost (native categorical handling)
- XGBoost with different hyperparameters (diversity)
- Neural Networks: TabM, RealMLP (if GPU available)

### Ensembling
- Weighted Average (OOF optimization)
- Stacking Level 2 (meta-model on OOF)
- Hill Climbing (weight optimization)
- Ridge Regression (stable blending)

### Final Pipeline
- Pseudo Labeling (confident test predictions added to train)
- Retrain on 100% data with +25% iterations
- Median of N seeds (5-10 seeds)